> **router_finetune_record.md 반영본(v3) — v2에서 사용자 피드백으로 3가지를 되돌림.** 원본(`train_router_moai_embedding.ipynb`)과 v2(`train_router_moai_embedding_v2_최종.ipynb`) 대비 이 파일에서 바뀐 점:
>
> 1. **에폭을 무조건 8개 다 돌리고 사후 선택하던 방식(§3-2-3) 되돌림** → 원본처럼 `EarlyStoppingCallback(patience=2)` + `load_best_model_at_end=True`(`metric_for_best_model="macro_f1"`)로 복귀. 별도 체크포인트 사후 선택 셀 삭제.
> 2. **OOS AUROC가 Energy/MSP 두 번 출력되던 것(§3-3 후반)을 원본처럼 MSP 기준 1회 출력으로 되돌림.** (단, G6 그룹을 AUROC 양성/음성 풀에서 제외하는 것 자체는 데이터 설계상 결정이라 유지 — 여기에 문제가 있다면 알려주세요.)
> 3. **학습셋 자동 분할(§3-1 제거분) 되돌림** → `VAL_PATH`가 없으면 원본처럼 학습셋을 80/20으로 자동 분할해서 epoch마다 `eval_accuracy`/`eval_macro_f1`이 출력되도록 복귀.
>
> `transformers==4.57.6` 고정·`accelerate>=1.1.0`·`warmup_ratio=0.1`·`LR=1e-5`·`batch 4×grad_accum4`·`bf16/fp16` 자동·`gradient_checkpointing` 등 §4 "MoAI 학습 상수"와 학습/테스트 데이터 파일 경로는 v2 그대로 유지했습니다(되돌려 달라는 요청이 없었던 부분).

---

# 라우터 후보 학습 — MoAI-Embedding (`BCCard/MoAI-Embedding-0.6B`)

**⚠️ 이 노트북은 리스크가 가장 높은 후보 중 하나 — 학습 성공을 전제하지 않는다**

- `BCCard/MoAI-Embedding-0.6B`는 `Qwen3Model`(디코더 전용, 0.6B) 백본이다. `model.safetensors`는 확인됐지만, 이 모델이 **임베딩 전용 모델**이라면 `AutoModelForSequenceClassification`이 기대하는 분류 헤드 결합 방식과 안 맞아 로딩 자체가 실패하거나(별도 어댑터 필요), 로딩은 되어도 학습이 의미 있게 되지 않을 수 있다.
- **이 파일의 1차 목적은 "학습 완주"가 아니라 "로딩·1 epoch 학습이 실제로 되는지"를 다른 8개 후보와 분리된 환경에서 확인하는 것.** 실패해도 정상적인 결과이며, 실패 시 이 후보는 그 자리에서 배제 판단의 근거가 된다.
- `Qwen3` 지원을 위해 `transformers>=4.51.0`이 필수 — 이 상한이 이전(4.44.2 고정) 노트북에서 다른 8개 후보(특히 ELECTRA류)와 충돌을 일으켰던 원인이었다. 이 파일은 이 후보 하나만 다루므로 그 충돌 자체가 사라진다.
- GPU/Colab 필요(Trainer 8-epoch 파인튜닝). `train_router_py.ipynb`(원본)는 이 작업으로 수정하지 않았다.

**실행 전 준비**: `DATA_DIR`(Colab `/content/data`)에 학습/검증/테스트 jsonl 파일 업로드.

In [ ]:
# transformers==4.57.6 고정, accelerate>=1.1.0 — router_finetune_record.md §4 확정값.
# (accelerate>=1.1.0은 0.33.0의 numpy<2 강제를 없애 numpy.rec 커널 오류의 원인을 제거한다.
#  다만 재시작 셀 자체는 원본 구조를 바꾸지 않기 위해 그대로 남겨둔다 — 실행해도 무해함.)
!pip install -q -U "transformers==4.57.6" "accelerate>=1.1.0" scikit-learn


**⚠️ 위 pip install 셀 실행 직후, 바로 아래 셀을 실행해 런타임을 강제 재시작하세요.**

`ModuleNotFoundError: No module named 'numpy.rec'` 같은 오류는 코드 문제가 아니라, 같은 커널 안에서 numpy 계열 패키지를 pip install한 뒤 재시작 없이 바로 import해서 생기는 코랩 특유의 증상이다(이미 로드된 numpy C-확장과 새로 깔린 numpy 바이너리가 어긋남). `transformers` 버전 점프가 크므로(4.44.2 → 4.51.0+) 이 파일에서는 재시작 셀을 생략하지 않는다. 재시작 후에는 pip install 셀을 다시 실행할 필요 없이 **다음 셀(임포트)부터 이어서 실행**한다.

In [ ]:
import os
os.kill(os.getpid(), 9)  # 코랩이 자동으로 새 런타임을 띄운다. 이후 다음 셀부터 이어서 실행.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# 원본 노트북(train_router_py.ipynb) 셀4 스키마 흡수 헬퍼 그대로.
def get_text(r: dict) -> str:
    return r.get("text") or r.get("question")


def get_label(r: dict):
    return r.get("business_function") or r.get("domain")


def is_oos(r: dict) -> bool:
    return bool(r.get("is_oos", False))


def is_g6(r: dict) -> bool:
    # G6(210건)은 설계상 임계 보정 전용 — OOS AUROC 풀에서 양성/음성 모두 제외(§3-3)
    return r.get("group") == "G6"


def load_jsonl(path: Path) -> list:
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("⚠️ GPU 미할당 — 0.6B 모델 학습은 CPU에서 비현실적으로 느립니다. 런타임 유형을 GPU로 변경하세요.")

In [ ]:
# 설정 — 원본 MODEL_CANDIDATES["moai-embedding"] 항목만 이 후보 전용 상수로 치환.
# (다른 8개 후보용 SELECTED_MODEL_KEY 분기·loader 가드는 이 파일에선 필요 없음.)

MODEL_NAME = "BCCard/MoAI-Embedding-0.6B"

DATA_DIR = Path("/content/data")
TRAIN_PATH = DATA_DIR / "router_trainset_v2_natural450_merged.jsonl"
VAL_PATH = DATA_DIR / "router_val.jsonl"                  # 없으면 TRAIN에서 자동 분할(VAL_RATIO)
TEST_PATH = DATA_DIR / "router_testset_v3_1.jsonl"    # MSP/AUROC 평가용, 선택

MAX_LEN = 64
# 원본 BATCH_SIZE=16(단일 값)을 router_finetune_record.md §4 "MoAI 학습 상수"의
# "batch 4 × grad_accum 4"로 교체 — 유효배치는 16으로 동일, full-FT 메모리 대응.
PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM_STEPS = 4
EPOCHS = 8             # EarlyStopping(patience=2)로 과적합 방어 — 원본처럼 실제로는 더 일찍 멈출 수 있음
LR = 1e-5              # §4: 0.6B full-FT에 2e-5는 붕괴 위험
WARMUP_RATIO = 0.1     # §4: 분류 헤드 랜덤 초기화 — warmup 없이 최대 LR 진입하던 문제 수정
WEIGHT_DECAY = 0.01
SEED = 42
VAL_RATIO = 0.2        # 원본처럼 복귀 — VAL_PATH가 없으면 학습셋을 80/20으로 자동 분할해 epoch별 accuracy를 본다.

OUTPUT_DIR = Path("/content/router_moai_embedding_out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print("selected model:", MODEL_NAME)

test_rows = load_jsonl(TEST_PATH)
print(f"test_rows={len(test_rows)}")

In [ ]:
# 데이터 로딩 (원본 셀6 그대로 — v2가 없앴던 자동 분할을 복귀)

train_rows = load_jsonl(TRAIN_PATH)
assert train_rows, (
    f"{TRAIN_PATH}가 비어있거나 없습니다. 학습 데이터 파일을 이 경로에 올린 뒤 다시 실행하세요."
)

val_rows = load_jsonl(VAL_PATH)
if not val_rows:
    random.Random(SEED).shuffle(train_rows)
    n_val = max(1, int(len(train_rows) * VAL_RATIO))
    val_rows, train_rows = train_rows[:n_val], train_rows[n_val:]

print(f"train={len(train_rows)}  val={len(val_rows)}  test(mixed)={len(test_rows)}")

In [ ]:
# 라벨 체계 (원본 셀7 그대로)

LABELS = sorted({get_label(r) for r in train_rows if get_label(r) and not is_oos(r)})
assert LABELS, "학습 데이터에 in-scope 라벨(business_function 또는 domain)이 없습니다."
label2id = {lab: i for i, lab in enumerate(LABELS)}
id2label = {i: lab for lab, i in label2id.items()}
print("LABELS:", LABELS)

train_rows = [r for r in train_rows if get_label(r) and not is_oos(r)]
val_rows = [r for r in val_rows if get_label(r) and not is_oos(r)]
print(f"학습에 실제 사용되는 in-scope 건수: train={len(train_rows)}  val={len(val_rows)}")

In [ ]:
# 토크나이즈 + Dataset (원본 셀8 그대로)
#
# moai-embedding(Qwen3 계열) 토크나이저는 기본적으로 pad_token이 없어서 그대로 두면
# padding="max_length"에서 에러가 난다. 이 모델에서는 이 분기가 실제로 작동한다.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


class RouterDataset(Dataset):
    def __init__(self, rows: list):
        texts = [get_text(r) for r in rows]
        self.labels = [label2id[get_label(r)] for r in rows]
        enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_mask[idx]),
            "labels": torch.tensor(self.labels[idx]),
        }


train_ds = RouterDataset(train_rows)
val_ds = RouterDataset(val_rows)
print(f"train_ds={len(train_ds)}  val_ds={len(val_ds)}")

In [ ]:
# 모델 로딩 (원본 셀9 그대로) — 이 셀이 이 노트북의 핵심 검증 지점이다.
# 임베딩 전용 모델이라 분류 헤드가 없다면 여기서 실패할 수 있다(정상적인 실패).

try:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)
except OSError as e:
    raise OSError(
        f"'{MODEL_NAME}' 로딩 실패 — 이 저장소에 pytorch_model.bin/model.safetensors 같은 "
        "가중치 파일이 없을 수 있습니다. HuggingFace 저장소 페이지에서 Files 탭을 직접 확인하세요."
    ) from e
except Exception as e:
    raise RuntimeError(
        f"'{MODEL_NAME}' 로딩 실패 — OSError가 아닌 다른 예외({type(e).__name__})입니다. "
        "이 모델이 임베딩 전용(분류 헤드 없음)이라 AutoModelForSequenceClassification과 구조가 "
        "안 맞을 가능성이 있습니다(알려진 리스크, 별도 어댑터 필요할 수 있음). "
        "이 예외 메시지 원문을 그대로 팀에 공유해 다음 판단 근거로 삼으세요."
    ) from e

# moai-embedding(Qwen3, 디코더 전용)은 분류 시 "마지막 non-pad 토큰" 위치를 config의
# pad_token_id로 찾는다 — 위에서 tokenizer.pad_token은 채웠지만 model.config에도 반영해야 한다.
model.config.pad_token_id = tokenizer.pad_token_id

model.config.use_cache = False  # §4 gradient_checkpointing과 함께 쓰려면 캐시를 꺼야 한다

# safetensors 저장 시 "non contiguous tensor" 오류 방지 — 미리 방어적으로 정리
for _p in model.parameters():
    _p.data = _p.data.contiguous()

print("모델 로딩 성공 — 이 지점을 통과했다는 것 자체가 1차 리스크(로딩 실패) 해소를 의미한다.")

In [ ]:
# 학습 — §4 "MoAI 학습 상수"(warmup_ratio·bf16/fp16 자동·gradient_checkpointing·batch/grad_accum)는 유지하되,
# 체크포인트 선택은 원본처럼 EarlyStopping + macro_f1 기준 load_best_model_at_end로 복귀(v2의 §3-2-3 사후선택 되돌림).
# → 학습이 끝나면 `model`은 이미 val macro_f1 기준 best 체크포인트로 자동 교체돼 있다. 별도 선택 셀 불필요.

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_ok = torch.cuda.is_available() and not bf16_ok
print(f"bf16={bf16_ok}  fp16={fp16_ok}")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


def build_training_args(**kw):
    try:
        return TrainingArguments(eval_strategy="epoch", **kw)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **kw)


training_args = build_training_args(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    per_device_eval_batch_size=PER_DEVICE_TRAIN_BATCH,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    bf16=bf16_ok,
    fp16=fp16_ok,
    gradient_checkpointing=True,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=10,
    seed=SEED,
    report_to="none",
    save_safetensors=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

train_result = trainer.train()
print(trainer.evaluate())

In [ ]:
# 평가 — MSP 기반 OOS + 안전성(LBR/AOC) + 교정(ECE) + 효율(APGR) 지표 (원본 evaluate_router 그대로)
# v2에서 Energy 스코어를 병행 출력해 AUROC가 2배로 찍히던 것을 원본처럼 MSP 1회 출력으로 되돌림.
# G6 그룹을 AUROC 풀(양성/음성)에서 빼는 것은 데이터 설계상 결정이라 유지함 — 원치 않으면 is_g6 필터만 지우면 됨.

BLOCK_TAU = 0.5   # 자리표시자 — 검증셋으로 재보정 필요
REVIEW_TAU = 0.8  # 자리표시자 — 재보정 필요


@torch.no_grad()
def predict_msp(rows: list):
    model.eval()
    texts = [get_text(r) for r in rows]
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs.argmax(axis=-1), probs.max(axis=-1)


def expected_calibration_error(confidences: np.ndarray, correct: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    n = len(confidences)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if not mask.any():
            continue
        ece += (mask.sum() / n) * abs(correct[mask].mean() - confidences[mask].mean())
    return float(ece)


def legitimate_block_rate(is_legit: np.ndarray, msp: np.ndarray, tau: float = BLOCK_TAU) -> float:
    blocked = msp < tau
    return float(blocked[is_legit].mean()) if is_legit.any() else float("nan")


def abstain_on_clean(is_legit: np.ndarray, msp: np.ndarray,
                      tau_block: float = BLOCK_TAU, tau_review: float = REVIEW_TAU) -> float:
    abstained = (msp >= tau_block) & (msp < tau_review)
    return float(abstained[is_legit].mean()) if is_legit.any() else float("nan")


def _trapz(y: np.ndarray, x: np.ndarray) -> float:
    dx = np.diff(x)
    return float(np.sum(dx * (y[:-1] + y[1:]) / 2))


def average_performance_gap_recovered(local_correct: np.ndarray, gpt4_correct, msp: np.ndarray, n_points: int = 21):
    if gpt4_correct is None:
        return None
    n = len(msp)
    call_rates = np.linspace(0.0, 1.0, n_points)
    local_acc, gpt4_acc = local_correct.mean(), gpt4_correct.mean()

    order = np.argsort(msp)
    benefit = gpt4_correct.astype(int) - local_correct.astype(int)
    oracle_order = np.argsort(-benefit)

    router_perf, oracle_perf, random_perf = [], [], []
    for cr in call_rates:
        k = int(round(cr * n))

        escalate = np.zeros(n, dtype=bool)
        escalate[order[:k]] = True
        router_perf.append(np.where(escalate, gpt4_correct, local_correct).mean())

        escalate_oracle = np.zeros(n, dtype=bool)
        escalate_oracle[oracle_order[:k]] = True
        oracle_perf.append(np.where(escalate_oracle, gpt4_correct, local_correct).mean())

        random_perf.append((1 - cr) * local_acc + cr * gpt4_acc)

    router_perf = np.array(router_perf)
    oracle_perf = np.array(oracle_perf)
    random_perf = np.array(random_perf)

    area_router = _trapz(router_perf - random_perf, call_rates)
    area_oracle = _trapz(oracle_perf - random_perf, call_rates)
    return float(area_router / area_oracle) if area_oracle > 0 else None


def evaluate_router(test_rows: list, gpt4_correct=None) -> None:
    if not test_rows:
        print("테스트셋이 비어 있어 MSP/AUROC 평가를 건너뜁니다.")
        return

    pred_idx, msp = predict_msp(test_rows)
    oos = np.array([is_oos(r) for r in test_rows])
    legit = ~oos

    in_scope_rows = [(i, r) for i, r in enumerate(test_rows)
                     if not is_oos(r) and get_label(r) in label2id]
    local_correct = None
    if in_scope_rows:
        idxs, rows_ = zip(*in_scope_rows)
        y_true = np.array([label2id[get_label(r)] for r in rows_])
        y_pred = np.array([pred_idx[i] for i in idxs])
        local_correct = (y_true == y_pred)
        print(f"[in-scope] n={len(y_true)}  accuracy={accuracy_score(y_true, y_pred):.3f}  "
              f"macro_f1={f1_score(y_true, y_pred, average='macro'):.3f}")
        print(confusion_matrix(y_true, y_pred))

        ece = expected_calibration_error(msp[np.array(idxs)], local_correct)
        print(f"[교정] ECE={ece:.3f} (in-scope, {len(y_true)}건)")
    else:
        print("[in-scope] 도메인 정확도/ECE skip — 학습 라벨(업무명)과 평가 라벨(D코드) 체계가 다름. "
              "D코드↔업무 매핑 확정 후 활성화.")

    # G6(210건)은 설계상 임계 보정 전용 — AUROC 양성/음성 풀 모두에서 제외(§3-3, v3에서도 유지)
    auroc_rows = [r for r in test_rows if not is_g6(r)]
    oos_a = np.array([is_oos(r) for r in auroc_rows])
    if oos_a.any() and (~oos_a).any():
        _, msp_a = predict_msp(auroc_rows)
        score_a = 1 - msp_a
        neg = ~oos_a
        print(f"[OOS 탐지] 전체 AUROC={roc_auc_score(oos_a, score_a):.3f}")
        for g in sorted({r.get("group") for r in auroc_rows if is_oos(r)}):
            pos = np.array([is_oos(r) and r.get("group") == g for r in auroc_rows])
            m = pos | neg
            if pos.sum() >= 1 and neg.sum() >= 1:
                tag = " <- near-OOD 승부처" if g == "G5b" else (" (far-OOD)" if g == "G5a" else "")
                print(f"   {g} AUROC={roc_auc_score(oos_a[m], score_a[m]):.3f}  (n_oos={int(pos.sum())}){tag}")
    else:
        print("dev(G6 제외)에 in-scope/OOS가 둘 다 있어야 AUROC를 계산할 수 있습니다.")

    if legit.any():
        lbr = legitimate_block_rate(legit, msp)
        aoc = abstain_on_clean(legit, msp)
        print(f"[안전성] LBR={lbr:.3f}  AOC={aoc:.3f}  "
              f"(tau_block={BLOCK_TAU}, tau_review={REVIEW_TAU} — 자리표시자, 재보정 필요)")

    if local_correct is not None:
        apgr = average_performance_gap_recovered(local_correct, gpt4_correct, msp[np.array(idxs)])
        if apgr is None:
            print("[효율] APGR skip — gpt4_correct(문항별 GPT-4 정답여부) 라벨이 없음.")
        else:
            print(f"[효율] APGR={apgr:.3f}")


evaluate_router(test_rows)

In [ ]:
# 모델 저장 (+ 코랩 밖으로 내보내기용 zip) — 원본 셀14 그대로

final_dir = OUTPUT_DIR / "final"
model.save_pretrained(final_dir, safe_serialization=False)
tokenizer.save_pretrained(final_dir)

import shutil
zip_path = shutil.make_archive(str(OUTPUT_DIR / "router_moai_embedding_model"), "zip", final_dir)
print("saved:", zip_path)

## 결과 해석 시 주의

- **모델 로딩(셀 "모델 로딩") 통과 여부가 1차 판단 기준.** 여기서 실패하면 이 후보는 임베딩 전용 구조와 `AutoModelForSequenceClassification`이 안 맞는다는 뜻이며, 별도 어댑터 없이는 이 방식으로 진행 불가 — 팀 논의로 넘길 것.
- 로딩·학습이 성공해도, 이 모델이 원래 임베딩(유사도) 목적으로 학습된 것이라 분류 태스크에서의 성능이 ELECTRA/DeBERTa 계열(kf-deberta)과 다른 양상을 보일 수 있음 — 단순 accuracy 비교보다 confusion matrix까지 함께 확인할 것.
- `PER_DEVICE_TRAIN_BATCH`(기본 4)는 GPU 메모리에 따라 조정 필요(0.6B는 ELECTRA류보다 훨씬 큼) — OOM 발생 시 2로 낮추고 `GRAD_ACCUM_STEPS`를 늘려 유효배치 16을 유지할 것.
- 이 결과는 kf-deberta 등 다른 hf_trainable 후보와 지표 체계(accuracy/macro_f1/MSP/AUROC/LBR/AOC/ECE/APGR)가 동일해 **직접 비교 가능**(intentguard/deycoding-router와는 불가).

## 이 v3 패치에서 v2 대비 바뀐 것 (사용자 피드백 반영)

- **체크포인트 선택**: v2의 "8 epoch 무조건 전부 학습 후 dev Energy AUROC로 사후 선택" 방식 제거 → 원본처럼 `EarlyStoppingCallback(patience=2)` + `load_best_model_at_end=True`(`metric_for_best_model="macro_f1"`)로 복귀. 별도 체크포인트 스코어링 셀 삭제 — `trainer.train()` 직후 `model`이 이미 best 체크포인트다.
- **OOS AUROC 출력**: v2가 Energy·MSP 두 스코어를 나란히 찍던 것 → 원본처럼 MSP 기준 1회 출력으로 복귀. (G6 그룹을 AUROC 풀에서 제외하는 로직 자체는 유지 — 필요 없으면 `evaluate_router` 안의 `is_g6` 필터만 지우면 원본과 100% 동일해짐.)
- **데이터 분할**: v2의 "학습셋 분할 없이 전량 사용" 제거 → 원본처럼 `VAL_PATH`가 없으면 `VAL_RATIO=0.2`로 자동 분할, 매 epoch마다 `eval_accuracy`/`eval_macro_f1`이 출력됨.
- **유지된 부분(되돌려 달라는 요청이 없었던 것)**: `transformers==4.57.6`·`accelerate>=1.1.0`·`warmup_ratio=0.1`·`LR=1e-5`·`batch 4×grad_accum4`·`bf16/fp16` 자동·`gradient_checkpointing`(§4 "MoAI 학습 상수"), 학습/테스트 데이터 파일 경로.
